In [1]:
import pandas as pd
from rdflib import Graph, Literal, Namespace, RDF, URIRef, OWL
from rdflib.namespace import DC, FOAF

from owlready2 import *

In [2]:
# load existing ontology   

onto = get_ontology("file://hi_ontology.owl").load()
hi = onto.get_namespace("http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/")

# check if ontology is sucessfully loaded
print("Base IRI:", onto.base_iri)
print(f"Classes in ontology: ")
for c in onto.classes():
  print(c)
print(f"Properties in ontology:")
for p in onto.properties():
  print(p)

Base IRI: http://www.semanticweb.org/vbr240/ontologies/2022/4/untitled-ontology-51/
Classes in ontology: 
hi_ontology.Human
hi_ontology.Scenario
hi_ontology.Context
hi_ontology.Endgoal
hi_ontology.Interaction
hi_ontology.InteractionTask
hi_ontology.InteractionMethod
hi_ontology.Actor
hi_ontology.InformationProcessing
hi_ontology.ProcessingMethod
hi_ontology.Capability
hi_ontology.ProcessingTask
hi_ontology.ArtificialAgent
hi_ontology.EthicalConsideration
hi_ontology.Domain
Properties in ontology:
hi_ontology.context
hi_ontology.endgoal
hi_ontology.interactionTask
hi_ontology.processingInformation
hi_ontology.interactingAgent
hi_ontology.capability
hi_ontology.informationMethod
hi_ontology.hasInteraction
hi_ontology.inScenario
hi_ontology.interactionMethod
hi_ontology.hasEthicalConsideration
hi_ontology.processingTask
hi_ontology.domain


### Paper 1: Trust in WareHouse Automation

**Purpose of the study**: Assess participants' trust in Automated Guieded Vehicles (AVGs) within mixed-fleet environment, specifically in warehouse (Context) settings.




**Assumption**

- Any instances are assigned to Worker classes represent for the whole group of actual workers with the same generation.

In [3]:
# create some classes:
class Worker(hi.Human):
    namespace = onto



# creates general instances
with onto:
  # scenario relevant
  scenario1 = onto.Scenario("WarehouseAVGscenario")
  domain1 = onto.Domain("WarehouseAutomation")
  context1 = onto.Context("TrustCalibration")
  context2 = onto.Context("GenerationalLevels")
  context3 = onto.Context("TaskComplexity")
  endgoal1 = onto.Endgoal("FillResearchGap")
  ethical1 = onto.EthicalConsideration("Privacy")
  ethical2 = onto.EthicalConsideration("Safety")

  # properties assertion
  scenario1.domain = [domain1]
  scenario1.context = [context1, context2, context3]
  scenario1.endgoal = [endgoal1]
  scenario1.hasEthicalConsideration = [ethical1, ethical2]

  # actor relevant
  # assumption worker represents for a whole group 
  # of workers have the same generation
  worker_genz = onto.Worker("Worker_GenZ")
  worker_geny = onto.Worker("Worker_GenY")
  worker_genx = onto.Worker("Worker_GenX")
  worker_genbabyboom = onto.Worker("Worker_GenBabyboom")

  agv = onto.ArtificialAgent("AGV_system")

  worker_genz.inScenario = [scenario1]
  worker_geny.inScenario = [scenario1]
  worker_genx.inScenario = [scenario1]
  worker_genbabyboom.inScenario = [scenario1]

  #  capability of agv
  agv.capability = [onto.Collaborativeness,
                    onto.Communication, 
                    onto.Transparency, # paper mentions as Predictability
                    onto.Adaptiveness]
  # interaction between worker and AGV
  interaction1 = onto.Interaction("Interaction_WorkerZ_AGV")
  interaction1.interactingAgent = [worker_genz, agv]

  interaction1.interactionTask = [onto.Explainability]


  interaction1.interactionMethod = [onto.Multimodal]

  scenario1.hasInteraction = [interaction1]

  # info processing 

  agvProcessing = onto.InformationProcessing("AGV_Processing")
  agvProcessing.processingTask = [onto.Reasoning, # reason over environment
                                  onto.Learning, # corrective actions
                                  onto.Transforming # transform into actions
    ]
  agvProcessing.informationMethod = [onto.Statistical] # only assumption
  agv.processingInformation = [agvProcessing]



In [4]:
# save 
onto.save(file="./onto_with_inst/hi_ontology_firstpaper.owl",
          format="rdfxml")


**Limitations**

- The levels of trust is not specified in the ontology while it is the main asset of the study.

### Paper 2: Human Response to Decision Support in Face Matching

In face matching (Scenario), a person is tasked with determining whether two photos correspond to the same person, and uses a face matching system as a decision support tool (Context).

**Purpose**: Investigate the influence (Endgoal) of task difficulty and machine precision on the human outcomes.

Domain = Face Matching

In [5]:
# function for connecting words with - in between
def get_by_fragment(fragment: str):
    ent = onto.search_one(iri=f"*{fragment}")
    if ent is None:
        raise ValueError(f"Not found in ontology: *{fragment}")
    return ent

In [6]:
# group of participants in one experimental setup


with onto:

  # general info about the scenario
  scenario2 = onto.Scenario("FaceMatchingDecisionSupport")
  domain2 = onto.Domain("FaceMatchingAI")
  context4 = onto.Context("TaskDifficulty")
  context5 = onto.Context("MachineAccuracy")
  context6 = onto.Context("HumanAwareness")
  endgoal2 = onto.Endgoal("ImproveHumanAccuracy")
  ethic3 = onto.EthicalConsideration("GenderFairness")



  scenario2.domain = [domain2]
  scenario2.context = [context4, context5, context6]
  scenario2.endgoal = [endgoal2]
  scenario2.hasEthicalConsideration = [ethic3]

  # actor specification
  # as the interaction is reported between 1 human vs 1 system
  participant1 = onto.Human("Participant1")
  dss = onto.ArtificialAgent("DecisionSupportSystem")

  participant1.inScenario = [scenario2]
  dss.inScenario = [scenario2]

  # As dss presents its label recommendation 
  # the transparency of dss is uncertain 
  # as used model is nn
  dss.capability = [onto.Communication]

  # information processing
  infoProcessing2 = onto.InformationProcessing("FaceMatchingLabeling")
  dss.processingInformation = [infoProcessing2]

  infoProcessing2.processingTask = [onto.Reasoning]
  infoProcessing2.processingMethod = [onto.Statistical]

  # interation 
  interaction2 = onto.Interaction("FaceMatchingLabelRecommendation")
  scenario2.hasInteraction = [interaction2]

  interaction2.interactingAgent = [participant1, dss]
  interaction2.interactionTask = [get_by_fragment("Question-Answering")]
  interaction2.interactionMethod = [get_by_fragment("Single-agent")] # only human active make decision

                

In [7]:
# save 
onto.save(file="./onto_with_inst/hi_ontology_first+secondpapers.owl",
          format="rdfxml")
